# 使用 Azure OpenAI 的语义内核与 Agent-to-Agent (A2A) 协议

本笔记本展示了如何使用语义内核与 A2A 协议，通过 Azure AI Foundry 和 Azure OpenAI 创建一个多代理的旅行规划系统。要设置您的环境变量，可以参考 [设置课程](https://github.com/microsoft/ai-agents-for-beginners/blob/main/translations/zh/00-course-setup/README.md)。

## 您将构建的内容

一个包含三个代理的旅行规划系统：
1. **货币兑换代理** - 使用实时汇率处理货币转换
2. **活动规划代理** - 规划活动并提供旅行建议
3. **旅行管理代理** - 协调其他代理以提供全面的旅行协助


## 安装

首先，让我们安装所需的依赖项：


## 导入所需库


In [1]:
import asyncio
import json
import logging
import os
import threading
import time
from typing import Any, Annotated, AsyncIterable, Literal
from enum import Enum

import httpx
import nest_asyncio
import uvicorn
from dotenv import load_dotenv
from pydantic import BaseModel

from openai import AsyncOpenAI

from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion  # OpenAI连接器
# A2A imports
# Add this to your imports at the top
from a2a.server.agent_execution import AgentExecutor
from a2a.client import ClientConfig, ClientFactory, create_text_message_object
from a2a.server.apps import A2AStarletteApplication
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore, InMemoryPushNotificationConfigStore, BasePushNotificationSender
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentSkill,
    TransportProtocol,
)
from a2a.utils.constants import AGENT_CARD_WELL_KNOWN_PATH

# Semantic Kernel imports
from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import (
    AzureChatCompletion,
    OpenAIChatPromptExecutionSettings,
)
from semantic_kernel.contents import (
    FunctionCallContent,
    FunctionResultContent,
    StreamingTextContent,
)
from semantic_kernel.functions import KernelArguments, kernel_function

## 环境配置

配置 Azure OpenAI 设置。确保已设置以下环境变量：
- `AZURE_OPENAI_CHAT_DEPLOYMENT_NAME`
- `AZURE_OPENAI_ENDPOINT`
- `AZURE_OPENAI_API_KEY`


In [2]:
# ======================
# 1. 环境配置
# ======================
# Load environment variables
load_dotenv()

# Apply nest_asyncio for running async code in Jupyter
# 确保Jupyter环境中能正确处理异步操作
nest_asyncio.apply()

# Setup logging
# 配置日志（方便调试）
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
)
logger = logging.getLogger(__name__)

print('Environment configured successfully!')


Environment configured successfully!


## 定义货币插件

此插件使用 Frankfurter API 提供实时货币汇率。


In [ ]:
# ======================
# 2. 货币插件定义
# ======================
class CurrencyPlugin:
    """A currency plugin that leverages Frankfurter API for exchange rates.
    一个使用Frankfurter API提供实时汇率的插件"""

    @kernel_function(# 使用Frankfurter API检索currency_from和currency_to之间的汇率
        description='使用Frankfurter API检索currency_from和currency_to之间的汇率'
    )
    def get_exchange_rate(
        self,
        currency_from: Annotated[str, '要转换的货币代码，例如 USD'],  # 要转换的货币（如USD）
        currency_to: Annotated[str, '目标货币代码，例如 EUR 或 INR'],   # 目标货币（如EUR）
        date: Annotated[str, "日期或 'latest'"] = 'latest',  # 日期（默认为最新）
    ) -> str:
        """获取两个货币之间的汇率"""
        try:
            # 调用Frankfurter API获取汇率
            response = httpx.get(
                f'https://api.frankfurter.app/{date}',
                params={'from': currency_from, 'to': currency_to},
                timeout=10.0,
            )
            response.raise_for_status() # 检查请求是否成功
            data = response.json()
            
            # 从返回数据中提取汇率
            if 'rates' not in data or currency_to not in data['rates']:
                return f'无法获取 {currency_from} 到 {currency_to} 的汇率。'
            rate = data['rates'][currency_to]
            return f'1 {currency_from} = {rate} {currency_to}'
        except Exception as e:
            return f"汇率接口调用失败：{str(e)}"

print('✅ Currency Plugin defined')

✅ Currency Plugin defined


## 定义响应格式

为代理输出定义结构化的响应格式。

### 目标

确保代理的输出清晰、一致且易于解析。

### 格式说明

以下是代理响应的标准格式：

1. **标题**: 提供一个简洁明了的标题，概述响应的主要内容。
2. **摘要**: 用一到两句话总结关键点。
3. **详细内容**: 提供更深入的解释或步骤。
4. **示例**: 包括代码片段或实际应用的例子（如果适用）。
5. **参考**: 提供相关链接或资源以供进一步阅读。

### 示例

以下是一个符合上述格式的示例：

- **标题**: 如何使用@@INLINE_CODE_1@@函数
- **摘要**: @@INLINE_CODE_1@@函数用于处理数据并返回结果。
- **详细内容**: 
  - 该函数接受两个参数：@@INLINE_CODE_2@@和@@INLINE_CODE_3@@。
  - 它会根据输入参数执行特定的逻辑。
- **示例**: 
  ```python
  result = @@INLINE_CODE_1@@(param1, param2)
  print(result)
  ```
- **参考**: 请参阅[官方文档](https://example.com)了解更多信息。

### 注意事项

- 始终遵循上述格式，确保输出的一致性。
- 避免使用模糊或不明确的语言。
- 如果某些部分不适用，可以省略，但应尽量保持完整性。

通过使用这种结构化格式，代理的输出将更具可读性和实用性。


In [8]:
# 定义一个名为 ResponseFormat 的类，它继承自 Pydantic 库的 BaseModel
# BaseModel 可以帮我们自动进行数据类型检查和格式校验
class ResponseFormat(BaseModel):
    
    # 这是一个多行注释（Docstring），说明这个类的用途：
    # "一个响应格式模型，用于指导（规范）大语言模型应该如何回复数据。"
    """一个响应格式模型，用于指导（规范）大语言模型应该如何回复数据。"""
    
    # 定义一个名为 status 的属性/字段
    # Literal['input_required', 'completed', 'error'] 表示大模型输出的 status 字段的值，必须且只能是这三个英文单词中的某一个，不能自由发挥
    # = 'input_required' 表示如果大模型没有返回这个字段，系统会默认把它设为 'input_required'
    status: Literal['input_required', 'completed', 'error'] = 'input_required'
    
    # 定义一个名为 message 的属性/字段
    # : str 限制了这个字段的内容必须是字符串（文本）类型
    message: str

# 打印一条带对勾表情的提示信息，告诉开发者这段类定义的代码已经成功运行完毕
print('✅ Response format defined')

✅ Response format defined


## 创建 A2A 代理执行器

此功能将语义内核代理封装为适用于 A2A 协议的形式。


In [ ]:
# ======================
# 3. A2A代理执行器
# ======================

# 这是一个"旅行规划总指挥"，负责协调其他代理
class SemanticKernelTravelAgentExecutor(AgentExecutor):
    """A2A Executor for Semantic Kernel Travel Agent.
    将Semantic Kernel代理封装为A2A协议兼容的执行器"""

    def __init__(self):
        """初始化方法，设置所有需要的组件
        - 创建与AI模型的连接
        - 创建三个专业代理：
            - 货币兑换代理（处理汇率问题）
            - 活动规划代理（规划旅行活动）
            - 旅行管理代理（总指挥，分配任务）"""

        # 创建OpenAI聊天服务（使用GitHub的Inference API）
        model_name = "gpt-4.1-mini"
        client = AsyncOpenAI(
            api_key=os.environ.get("GITHUB_TOKEN"), 
            base_url="https://models.inference.ai.azure.com/",
        )
        self.chat_service = OpenAIChatCompletion(
            ai_model_id=model_name,
            async_client=client,
        )

        # Create Currency Exchange Agent
        # 创建货币兑换代理
        self.currency_agent = ChatCompletionAgent(
            service=self.chat_service,
            name='CurrencyExchangeAgent',
            instructions=(
                '你是一名专门处理旅行者货币相关问题的助手。'
                '你的职责包括提供最新的汇率信息、在不同货币之间进行金额换算、'
                '解释货币兑换过程中可能产生的手续费或其他费用，并为用户提供货币兑换的最佳实践建议。'
                '你的目标是快速、准确地帮助旅行者解决所有与货币相关的问题。'
            ),
            plugins=[CurrencyPlugin()],
        )

        # Create Activity Planner Agent
        # 创建活动规划代理
        self.activity_agent = ChatCompletionAgent(
            service=self.chat_service,
            name='ActivityPlannerAgent',
            instructions=(
                "你是一名专门为旅行者规划和推荐旅行活动的助手。"
                "你的职责包括推荐观光景点、本地特色活动、美食餐厅，"
                "协助预订景点门票、制定旅行行程，并确保所有推荐的活动都符合旅行者的兴趣偏好和时间安排。"
                "你的目标是为旅行者打造愉快、个性化且难忘的旅行体验。"
            ),
        )

        # Create the main Travel Manager Agent - simplified approach
        # 创建旅行管理代理（主协调器）
        self.travel_agent = ChatCompletionAgent(
            service=self.chat_service,
            name='TravelManagerAgent',
            instructions=(
                "你的职责是认真分析旅行者的请求，并根据请求的具体内容，将其转交给最合适的智能体（Agent）处理。"
                "凡是涉及金额、汇率查询、货币兑换、货币兑换手续费、金融交易或支付方式等相关请求，"
                "都应转交给 CurrencyExchangeAgent（货币兑换助手）处理。"
                "凡是涉及旅行活动规划、景点推荐、美食推荐、活动预订、行程制定，"
                "或其他与旅行体验相关但不直接涉及金钱交易的请求，"
                "都应转交给 ActivityPlannerAgent（活动规划助手）处理。"
                "你的首要目标是准确、高效地完成请求分发，确保旅行者能够及时获得专业且准确的帮助。"
            ),
            plugins=[self.currency_agent, self.activity_agent],
        )

        self.thread = None  # 用于存储对话历史
        self.SUPPORTED_CONTENT_TYPES = ['text', 'text/plain']  # 支持的内容类型

    async def execute(self, context, event_queue):
        """Execute method required by A2A framework.
        A2A协议要求的执行方法（核心逻辑），由A2A框架调用
        执行方法，当收到用户请求时调用
        1、获取用户输入
        2、确保对话历史存在
        3、让旅行管理代理处理请求
        4、将结果发送回客户端
        """
        try:
            # Import required A2A utilities
            from a2a.utils import new_agent_text_message, new_task, new_text_artifact
            from a2a.types import TaskArtifactUpdateEvent, TaskState, TaskStatus, TaskStatusUpdateEvent

            # Get user input using the correct context method
            # 获取用户输入
            user_input = context.get_user_input()
            task = context.current_task

            # 确保任务存在
            if not task:
                task = new_task(context.message)
                await event_queue.enqueue_event(task)

            # Ensure thread exists
            # 确保对话线程存在
            session_id = task.context_id
            await self._ensure_thread_exists(session_id)

            # Process the request - no special response format, let it respond naturally
            # 处理请求 - 让旅行管理代理分析并回复
            response = await self.travel_agent.get_response(
                messages=user_input,
                thread=self.thread,
            )

            # Get the content directly as string
            # 获取内容（确保是字符串）
            content = response.content if isinstance(response.content, str) else str(response.content)

            # Send completion event with artifact
            # 发送结果事件（告诉客户端有结果了）
            # 第一个调用发送TaskArtifactUpdateEvent：这表示任务产生了结果（artifact），即代理的响应内容。
            # last_chunk=True表示这是最后一个数据块（完整响应）。
            await event_queue.enqueue_event(
                TaskArtifactUpdateEvent(
                    append=False,
                    context_id=task.context_id,
                    task_id=task.id,
                    last_chunk=True,
                    artifact=new_text_artifact(
                        name='travel_result',
                        description='行程规划结果',
                        text=content,
                    ),
                )
            )
            
            # 发送完成状态事件
            # 第二个调用发送TaskStatusUpdateEvent：这表示任务状态更新为"completed"（已完成）。
            await event_queue.enqueue_event(
                TaskStatusUpdateEvent(
                    status=TaskStatus(state=TaskState.completed),
                    final=True,
                    context_id=task.context_id,
                    task_id=task.id,
                )
            )

        except Exception as e:
            logger.error(f"Error in SemanticKernelTravelAgentExecutor.execute: {str(e)}")
            # Send error status
            # 发送错误状态
            await event_queue.enqueue_event(
                TaskStatusUpdateEvent(
                    status=TaskStatus(
                        state=TaskState.input_required,
                        message=new_agent_text_message(
                            f"Error processing request: {str(e)}",
                            task.context_id,
                            task.id,
                        ),
                    ),
                    final=True,
                    context_id=task.context_id,
                    task_id=task.id,
                )
            )

    async def cancel(self, context, event_queue):
        """Cancel method - not supported for this agent."""
        """取消方法 - 当前不支持"""
        raise Exception('不支持取消')

    async def _ensure_thread_exists(self, session_id: str) -> None:
        """Ensure thread exists for the session."""
        """确保对话线程存在"""
        # 如果线程不存在或ID不匹配，创建新线程
        if self.thread is None or self.thread.id != session_id:
            if self.thread:
                await self.thread.delete() # 删除旧线程
            self.thread = ChatHistoryAgentThread(thread_id=session_id)  # 创建新线程


print('✅ 旅行管理代理执行器简化 - 移除了JSON格式化约束')

✅ Travel Manager Agent Executor simplified - removed JSON formatting constraints


## 创建单独的 A2A 代理

现在我们将为每个专门代理创建 A2A 包装器。


In [12]:
# 货币代理执行器（专门处理货币问题的代理）
class CurrencyAgentExecutor(AgentExecutor):
    """A2A Executor for Currency Exchange Agent.
    货币兑换代理的A2A执行器"""

    def __init__(self):
        # self.chat_service = AzureChatCompletion(
        #     deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
        #     endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        #     api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        # )
        # 创建OpenAI聊天服务（使用GitHub的Inference API）
        model_name="qwen-max"
        client = AsyncOpenAI(
            api_key=os.environ.get("DASHSCOPE_API_KEY"), 
            base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
        )
        # model_name = "gpt-4o-mini"
        # client = AsyncOpenAI(
        #     api_key=os.environ["GITHUB_TOKEN"],
        #     base_url="https://models.inference.ai.azure.com/"
        # )
        self.chat_service = OpenAIChatCompletion(
            # 指定要使用的模型ID（这里是通义千问的qwen-max）
            ai_model_id=model_name,
            # 传入之前创建的AsyncOpenAI客户端
            async_client=client,
        )

        # 创建专门的货币代理
        # 提示词翻译如下：
        # 您是货币兑换专家。提供准确的汇率和货币转换信息。
        # 使用get_exchange_rate函数获取实时汇率。
        # 始终提供清晰简明的货币转换信息。
        self.agent = ChatCompletionAgent(
            service=self.chat_service,
            name='CurrencyExchangeAgent',
            instructions=(
                'You are a currency exchange specialist. Provide accurate exchange rates and currency conversion information. '
                'Use the get_exchange_rate function to get real-time rates. '
                'Always provide clear, concise information about currency conversions.'
            ),
            plugins=[CurrencyPlugin()],
        )
        self.thread = None
        self.SUPPORTED_CONTENT_TYPES = ['text', 'text/plain']

    async def execute(self, context, event_queue):
        """Execute method required by A2A framework.
        与主执行器类似，但仅处理单一代理逻辑
        主要区别：只调用货币代理，不涉及任务分配
        """
        try:
            # Import required A2A utilities
            from a2a.utils import new_agent_text_message, new_task, new_text_artifact
            from a2a.types import TaskArtifactUpdateEvent, TaskState, TaskStatus, TaskStatusUpdateEvent

            # Get user input using the correct context method
            user_input = context.get_user_input()
            task = context.current_task
            
            if not task:
                task = new_task(context.message)
                await event_queue.enqueue_event(task)

            # Ensure thread exists
            session_id = task.context_id
            if self.thread is None or self.thread.id != session_id:
                if self.thread:
                    await self.thread.delete()
                self.thread = ChatHistoryAgentThread(thread_id=session_id)

            # Process the request
            response = await self.agent.get_response(messages=user_input, thread=self.thread)
            content = response.content if isinstance(response.content, str) else str(response.content)

            # Send completion event with artifact
            await event_queue.enqueue_event(
                TaskArtifactUpdateEvent(
                    append=False,
                    context_id=task.context_id,
                    task_id=task.id,
                    last_chunk=True,
                    artifact=new_text_artifact(
                        name='currency_result',
                        description='Currency exchange information',
                        text=content,
                    ),
                )
            )
            
            await event_queue.enqueue_event(
                TaskStatusUpdateEvent(
                    status=TaskStatus(state=TaskState.completed),
                    final=True,
                    context_id=task.context_id,
                    task_id=task.id,
                )
            )

        except Exception as e:
            logger.error(f"Error in CurrencyAgentExecutor.execute: {str(e)}")
            # Send error status
            await event_queue.enqueue_event(
                TaskStatusUpdateEvent(
                    status=TaskStatus(
                        state=TaskState.input_required,
                        message=new_agent_text_message(
                            f"Error processing request: {str(e)}",
                            task.context_id,
                            task.id,
                        ),
                    ),
                    final=True,
                    context_id=task.context_id,
                    task_id=task.id,
                )
            )

    async def cancel(self, context, event_queue):
        """Cancel method - not supported for this simple agent."""
        raise Exception('cancel not supported')

# Updated Activity Planner Agent Executor
# 活动代理执行器（专门处理活动规划的代理）
class ActivityAgentExecutor(AgentExecutor):
    """A2A Executor for Activity Planner Agent."""

    def __init__(self):
        # self.chat_service = AzureChatCompletion(
        #     deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
        #     endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        #     api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        # )
        # 创建OpenAI聊天服务（使用GitHub的Inference API）
        model_name="qwen-max"
        client = AsyncOpenAI(
            api_key=os.environ.get("DASHSCOPE_API_KEY"), 
            base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
        )
        # model_name = "gpt-4o-mini"
        # client = AsyncOpenAI(
        #     api_key=os.environ["GITHUB_TOKEN"],
        #     base_url="https://models.inference.ai.azure.com/"
        # )
        self.chat_service = OpenAIChatCompletion(
            # 指定要使用的模型ID（这里是通义千问的qwen-max）
            ai_model_id=model_name,
            # 传入之前创建的AsyncOpenAI客户端
            async_client=client,
        )

        # 创建专门的活动规划代理
        # 提示词翻译如下：
        # 您是旅行活动规划专家。创建详细的个性化活动推荐。
        # 包括具体的时间、地点和实用提示。
        # 在建议中考虑预算、偏好和当地文化。  
        self.agent = ChatCompletionAgent(
            service=self.chat_service,
            name='ActivityPlannerAgent',
            instructions=(
                'You are a travel activity planning specialist. Create detailed, personalized activity recommendations. '
                'Include specific times, locations, and practical tips. '
                'Consider budget, preferences, and local culture in your suggestions.'
            ),
        )
        self.thread = None
        self.SUPPORTED_CONTENT_TYPES = ['text', 'text/plain']

    async def execute(self, context, event_queue):
        """Execute method required by A2A framework."""
        try:
            # Import required A2A utilities
            from a2a.utils import new_agent_text_message, new_task, new_text_artifact
            from a2a.types import TaskArtifactUpdateEvent, TaskState, TaskStatus, TaskStatusUpdateEvent

            # Get user input using the correct context method
            user_input = context.get_user_input()
            task = context.current_task
            
            if not task:
                task = new_task(context.message)
                await event_queue.enqueue_event(task)

            # Ensure thread exists
            session_id = task.context_id
            if self.thread is None or self.thread.id != session_id:
                if self.thread:
                    await self.thread.delete()
                self.thread = ChatHistoryAgentThread(thread_id=session_id)

            # Process the request
            response = await self.agent.get_response(messages=user_input, thread=self.thread)
            content = response.content if isinstance(response.content, str) else str(response.content)

            # Send completion event with artifact
            await event_queue.enqueue_event(
                TaskArtifactUpdateEvent(
                    append=False,
                    context_id=task.context_id,
                    task_id=task.id,
                    last_chunk=True,
                    artifact=new_text_artifact(
                        name='activity_result',
                        description='Activity planning recommendations',
                        text=content,
                    ),
                )
            )
            
            await event_queue.enqueue_event(
                TaskStatusUpdateEvent(
                    status=TaskStatus(state=TaskState.completed),
                    final=True,
                    context_id=task.context_id,
                    task_id=task.id,
                )
            )

        except Exception as e:
            logger.error(f"Error in ActivityAgentExecutor.execute: {str(e)}")
            # Send error status
            await event_queue.enqueue_event(
                TaskStatusUpdateEvent(
                    status=TaskStatus(
                        state=TaskState.input_required,
                        message=new_agent_text_message(
                            f"Error processing request: {str(e)}",
                            task.context_id,
                            task.id,
                        ),
                    ),
                    final=True,
                    context_id=task.context_id,
                    task_id=task.id,
                )
            )

    async def cancel(self, context, event_queue):
        """Cancel method - not supported for this simple agent."""
        raise Exception('cancel not supported')

## 定义代理卡片

代理卡片描述了每个代理在A2A发现中的能力。


In [13]:
# Currency Exchange Agent Card
# 货币代理卡片 - 描述货币代理的能力
currency_agent_card = AgentCard(
    name='Currency Exchange Agent',  # 代理名称
    url='http://localhost:10020',  # 代理地址
    description='Provides real-time currency exchange rates and conversion services', # 代理描述
    version='1.0',  # 版本号
    capabilities=AgentCapabilities(streaming=True),  # 支持流式响应
    default_input_modes=['text/plain'],  # 默认输入格式
    default_output_modes=['text/plain'],  # 默认输出格式
    preferred_transport=TransportProtocol.jsonrpc, # 首选通信协议
    skills=[  # 代理技能
        AgentSkill(
            id='currency_exchange', # 技能ID
            name='Currency Exchange', # 技能名称 货币兑换
            description='Get exchange rates and convert between currencies', # 技能描述 获取汇率并在货币之间转换
            tags=['currency', 'exchange', 'conversion', 'forex'], # 标签
            examples=[ # 使用示例
                'What is the exchange rate from USD to EUR?',  # USD到EUR的汇率是多少？
                'Convert 1000 USD to JPY',  # 将1000美元转换为日元
                'How much is 500 EUR in GBP?',  # 500欧元等于多少英镑？
            ],
        )
    ],
)

# Activity Planner Agent Card
# 活动规划代理卡片 - 描述活动规划代理的能力
activity_agent_card = AgentCard(
    name='Activity Planner Agent',  #  活动规划代理
    url='http://localhost:10021',
    description='Plans activities and provides travel recommendations',  # 规划活动并提供旅行建议
    version='1.0',
    capabilities=AgentCapabilities(streaming=True),
    default_input_modes=['text/plain'],
    default_output_modes=['text/plain'],
    preferred_transport=TransportProtocol.jsonrpc,
    skills=[
        AgentSkill(
            id='activity_planning',
            name='Activity Planning',
            description='Create personalized travel itineraries and activity recommendations',  # 创建个性化旅行行程和活动推荐
            tags=['travel', 'activities', 'itinerary', 'recommendations'],  
            examples=[
                'Plan a day trip in Paris',  # 在巴黎规划一日游
                'Recommend restaurants in Tokyo',  # 推荐东京的餐厅
                'What are the must-see attractions in Rome?',  # 罗马必看的景点有哪些？
            ],
        )
    ],
)

# Travel Manager Agent Card (Main orchestrator)
travel_manager_card = AgentCard(
    name='SK Travel Manager',
    url='http://localhost:10022',
    description='Comprehensive travel planning agent that orchestrates currency and activity services',  # 综合旅行规划代理，协调货币和活动服务
    version='1.0',
    capabilities=AgentCapabilities(streaming=True),
    default_input_modes=['text/plain'],
    default_output_modes=['text/plain'],
    preferred_transport=TransportProtocol.jsonrpc,
    skills=[
        AgentSkill(
            id='comprehensive_travel_planning',
            name='Comprehensive Travel Planning',  # 综合旅行规划
            description='Handles all aspects of travel planning including currency and activities',  # 处理旅行规划的各个方面，包括货币和活动
            tags=['travel', 'planning', 'currency', 'activities', 'orchestration'],
            examples=[
                'Plan a budget-friendly trip to Seoul with currency exchange info',  # 规划一次经济实惠的首尔之旅，并提供货币兑换信息
                'I need help with my Tokyo trip including money exchange and activities',  # 我需要帮助规划东京之旅，包括货币兑换和活动
                'What should I do in London and how much money should I exchange?',  # 我在伦敦应该做什么，我应该兑换多少钱？
            ],
        )
    ],
)

print('✅ Agent cards defined')

✅ Agent cards defined


## 创建A2A服务器辅助函数


此函数通过设置HTTP通信基础设施、配置带有任务管理和推送通知的请求处理器、将所有内容封装在Starlette Web应用程序中，并返回可运行的服务器实例，从而创建一个A2A（Agent-to-Agent）协议服务器。

A2AStarletteApplication是一个基于Starlette ASGI框架构建的Web应用程序封装器，它实现了A2A协议标准，允许AI代理通过HTTP使用标准化的消息格式和发现机制进行通信。


In [14]:


def create_a2a_server(agent_executor, agent_card):
    """Create an A2A server for any agent executor.
    创建A2A服务器的辅助函数
    """
    # 创建HTTP客户端
    httpx_client = httpx.AsyncClient()
    
    # 创建内存中的任务存储（用于保存任务状态）
    push_config_store = InMemoryPushNotificationConfigStore()

    # 创建请求处理器（处理进来的请求）
    # 实例化一个 DefaultRequestHandler（默认请求处理器）对象，并赋值给 request_handler 变量
    # 这个处理器通常负责接收用户的输入/请求，并协调内部的各个组件来完成任务
    request_handler = DefaultRequestHandler(    
        # 传入 agent_executor（智能体执行器）
        # 它是 AI 系统的“大脑”，负责调用大模型、思考逻辑以及执行外部工具（Tools）
        agent_executor=agent_executor,       
        # 传入 task_store（任务存储组件），并实例化为一个 InMemoryTaskStore（内存任务存储）
        # 用于保存当前正在处理的任务状态或历史记录。InMemory 表示数据暂存在运行内存中，重启程序后数据会清空
        task_store=InMemoryTaskStore(),       
        # 传入 push_config_store（推送配置存储）
        # 用于管理和读取向用户发送推送通知的相关配置（比如用户的设备 Token、推送开关等）
        push_config_store=push_config_store,       
        # 传入 push_sender（推送发送器），并实例化为一个 BasePushNotificationSender（基础推送通知发送器）
        # 它的作用是在任务完成或有状态更新时，真正把消息推送给用户端（如手机 App 或网页）
        push_sender=BasePushNotificationSender(          
            # BasePushNotificationSender 需要两个参数来完成工作：
            # 1. httpx_client：一个异步 HTTP 客户端，用来向苹果(APNs)、谷歌(FCM)或微信等第三方服务器发送网络请求
            httpx_client,           
            # 2. push_config_store：再次传入推送配置存储，这样发送器在发送前就能查到该发给谁、怎么发
            push_config_store
        ),
    )

    # 创建A2A应用
    app = A2AStarletteApplication(
        agent_card=agent_card,  # 代理卡片
        http_handler=request_handler  # HTTP处理器
    )

    # Return the actual Starlette app
    # Check if we need to build or get the app
    # 返回实际的Starlette应用
    # 检查不同版本的A2A库如何获取应用对象
    if hasattr(app, 'build'):
        return app.build()
    elif hasattr(app, 'app'):
        return app.app
    else:
        return app


print('✅ A2A server helper function created')

✅ A2A server helper function created


## 启动所有 A2A 服务器

我们将使用 uvicorn 将这三个代理分别作为独立的 A2A 服务器运行。


In [ ]:
# ======================
# 4. A2A服务启动
# ======================

async def run_agent_server(agent_executor, agent_card, port):
    """Run a single agent server with proper error handling.
    在后台运行单个代理服务"""
    try:
        # 创建A2A服务器
        app = create_a2a_server(agent_executor, agent_card)

        # 配置服务器
        config = uvicorn.Config(
            app,
            host='127.0.0.1',  # 本地主机
            port=port,  # 指定端口
            log_level='info',  # 日志级别
            loop='none',  # 事件循环
            timeout_keep_alive=30,  # 保持连接30秒
            limit_concurrency=100,  # 最大并发连接数
        )

        # 创建并启动服务器
        # Uvicorn是一个高性能的ASGI（异步服务器网关接口）服务器，用于运行Python异步Web应用。
        server = uvicorn.Server(config)
        await server.serve()
    except Exception as e:
        logger.error(f"Error starting server on port {port}: {str(e)}")
        raise


# ...existing code...

# Global variable to track running servers
running_servers = []


async def start_all_servers_background():
    """Start all servers in background tasks."""
    global running_servers

    try:
        # Create agent executors
        # 创建三个代理执行器
        currency_executor = CurrencyAgentExecutor()
        activity_executor = ActivityAgentExecutor()
        travel_executor = SemanticKernelTravelAgentExecutor()

        # Create tasks for all servers
        # 为每个代理创建服务器任务
        tasks = [
            asyncio.create_task(run_agent_server(
                currency_executor, currency_agent_card, 10020)),
            asyncio.create_task(run_agent_server(
                activity_executor, activity_agent_card, 10021)),
            asyncio.create_task(run_agent_server(
                travel_executor, travel_manager_card, 10022)),
        ]

        running_servers = tasks  # 保存运行中的服务器

        # Give servers time to start
        await asyncio.sleep(3)

        print('✅ 所有A2A代理服务器已在后台启动！')
        print('   - 货币兑换代理: http://127.0.0.1:10020')
        print('   - 活动规划代理: http://127.0.0.1:10021')
        print('   - 旅行管理代理: http://127.0.0.1:10022')

        # Don't await the tasks here - let them run in background
        return tasks

    except Exception as e:
        logger.error(f"Error in start_all_servers: {str(e)}")
        print(f"Failed to start servers: {str(e)}")
        raise

# Start the servers in background
server_tasks = await start_all_servers_background()

INFO:     Started server process [76161]
INFO:     Waiting for application startup.
INFO:     Started server process [76161]
INFO:     Waiting for application startup.
INFO:     Started server process [76161]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Application startup complete.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:10020 (Press CTRL+C to quit)
INFO:     Uvicorn running on http://127.0.0.1:10021 (Press CTRL+C to quit)
INFO:     Uvicorn running on http://127.0.0.1:10022 (Press CTRL+C to quit)


✅ 所有A2A代理服务器已在后台启动！
   - 货币兑换代理: http://127.0.0.1:10020
   - 活动规划代理: http://127.0.0.1:10021
   - 旅行管理代理: http://127.0.0.1:10022


INFO:     127.0.0.1:50838 - "GET / HTTP/1.1" 405 Method Not Allowed
INFO:     127.0.0.1:50838 - "GET /favicon.ico HTTP/1.1" 404 Not Found


In [24]:
# Add this cell after starting the servers but before testing
async def verify_servers():
    """Verify that all A2A servers are running and accessible.
    验证所有A2A服务器是否正常运行"""
    import httpx

    servers = [
        ('Currency Exchange Agent', 'http://localhost:10020'),
        ('Activity Planner Agent', 'http://localhost:10021'),
        ('Travel Manager Agent', 'http://localhost:10022'),
    ]

    print("🔍 Verifying A2A servers...")
    print("="*50)

    async with httpx.AsyncClient() as client:
        for name, url in servers:
            try:
                # 检查代理卡片端点（标准A2A协议端点）
                response = await client.get(f"{url}{AGENT_CARD_WELL_KNOWN_PATH}", timeout=5.0)
                if response.status_code == 200:
                    print(f"✅ {name} is running at {url}")
                else:
                    print(f"⚠️ {name} returned status {response.status_code}")
            except Exception as e:
                print(f"❌ {name} is not accessible: {str(e)}")

    print("="*50)

# Run server verification
await verify_servers()


🔍 Verifying A2A servers...
INFO:     127.0.0.1:53745 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK


2026-05-03 18:56:44,057 - INFO - httpx - HTTP Request: GET http://localhost:10020/.well-known/agent-card.json "HTTP/1.1 200 OK"


✅ Currency Exchange Agent is running at http://localhost:10020
INFO:     127.0.0.1:53747 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK


2026-05-03 18:56:44,060 - INFO - httpx - HTTP Request: GET http://localhost:10021/.well-known/agent-card.json "HTTP/1.1 200 OK"


✅ Activity Planner Agent is running at http://localhost:10021
INFO:     127.0.0.1:53749 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK


2026-05-03 18:56:44,064 - INFO - httpx - HTTP Request: GET http://localhost:10022/.well-known/agent-card.json "HTTP/1.1 200 OK"


✅ Travel Manager Agent is running at http://localhost:10022


## 创建 A2A 客户端

现在让我们创建一个客户端来与我们的 A2A 代理进行交互。

- `A2AClient`：就像一个"代理访问器"，让我们可以与前面启动的服务器通信
- 工作流程：
    1. 检查代理信息（先看缓存，没有就从服务器获取）
    2. 创建与代理通信的客户端
    3. 发送消息给代理
    4. 接收并解析代理的响应
- 关键特点：
    - 使用缓存：避免重复请求代理信息
    - 超时设置：防止程序卡死
    - 增强型解析：尝试多种方式从响应中提取文本
- 就像你打电话给客服：
    1. 先查客服电话号码（代理信息）
    2. 拨打电话（创建连接）
    3. 说出你的问题（发送消息）
    4. 听客服回答（接收响应）
    5. 理解客服说的话（解析响应）

In [17]:
class A2AClient:
    """Simple A2A client to interact with A2A servers.
    与A2A服务器交互的简单客户端"""
    
    def __init__(self, default_timeout: float = 60.0):
        """初始化客户端
        default_timeout: 默认超时时间（秒），防止程序卡死"""
        # 代理信息缓存 - 避免重复请求相同的代理信息
        # 就像通讯录，记住每个代理的"电话号码"和"介绍"
        self._agent_info_cache = {}
        
        # 默认超时时间 - 与代理通信最多等待的时间
        self.default_timeout = default_timeout
    
    async def send_message(self, agent_url: str, message: str) -> str:
        """Send a message to an A2A agent.
        向指定的A2A代理发送消息并获取响应
        agent_url: 代理地址（如'http://localhost:10020'）
        message: 要发送的消息内容
        返回: 代理的响应文本"""
        timeout_config = httpx.Timeout(
            timeout=self.default_timeout,  # 整个请求的总超时
            connect=10.0,  # 连接服务器最多等待10秒
            read=self.default_timeout,  # 读取响应最多等待default_timeout秒
            write=10.0,  # 发送请求最多等待10秒
            pool=5.0,  # 从连接池获取连接最多等待5秒
        )
        
        # 创建HTTP客户端（带超时设置），使用async with确保资源正确释放
        async with httpx.AsyncClient(timeout=timeout_config) as httpx_client:
            # Fetch agent card if not cached
            # 1. 获取代理信息（先检查缓存）
            if agent_url not in self._agent_info_cache:
                # 请求标准A2A端点获取代理卡片
                agent_card_response = await httpx_client.get(
                    f'{agent_url}{AGENT_CARD_WELL_KNOWN_PATH}'
                )
                # 将响应保存到缓存，避免重复请求
                self._agent_info_cache[agent_url] = agent_card_response.json()
            
            # 2. 解析代理卡片数据
            agent_card_data = self._agent_info_cache[agent_url]
            # 将JSON数据转换为AgentCard对象
            agent_card = AgentCard(**agent_card_data)
            
            # Create A2A client
            # 3. 创建A2A客户端配置
            config = ClientConfig(
                # 使用当前HTTP客户端
                httpx_client=httpx_client,
                # 支持的通信协议（JSON-RPC和HTTP-JSON）
                supported_transports=[
                    TransportProtocol.jsonrpc,
                    TransportProtocol.http_json,
                ],
                # 使用客户端偏好（而不是服务器偏好）
                use_client_preference=True,
            )
            
            # 4. 创建客户端工厂并生成具体客户端
            factory = ClientFactory(config)
            # 根据代理卡片创建能与该代理通信的客户端
            client = factory.create(agent_card)
            
            # 5. 创建消息对象（符合A2A协议格式）
            message_obj = create_text_message_object(content=message)
            
            # 6. 发送消息并收集所有响应
            responses = []
            # A2A支持流式响应，所以使用async for接收所有片段
            async for response in client.send_message(message_obj):
                responses.append(response)
            
            # Enhanced response parsing
            # 7. 增强型响应解析（尝试多种方式提取文本）
            if responses:
                try:
                    # The response should be a tuple (task, any_additional_data)
                    # 尝试多种解析路径，确保兼容不同响应结构
                    for response_item in responses:
                        if isinstance(response_item, tuple) and len(response_item) > 0:
                            task = response_item[0]
                            
                            # Try multiple ways to extract the response text
                            if hasattr(task, 'artifacts') and task.artifacts:
                                for artifact in task.artifacts:
                                    if hasattr(artifact, 'parts') and artifact.parts:
                                        for part in artifact.parts:
                                            if hasattr(part, 'root') and hasattr(part.root, 'text'):
                                                return part.root.text
                                            elif hasattr(part, 'text'):
                                                return part.text
                                            elif hasattr(part, 'content'):
                                                return str(part.content)
                                    elif hasattr(artifact, 'text'):
                                        return artifact.text
                                    elif hasattr(artifact, 'content'):
                                        return str(artifact.content)
                            
                            # If artifacts don't work, try direct task properties
                            # 尝试直接从任务对象获取文本
                            elif hasattr(task, 'text'):
                                return task.text
                            elif hasattr(task, 'content'):
                                return str(task.content)
                            elif hasattr(task, 'result'):
                                return str(task.result)
                            else:
                                # Debug: print the task structure
                                print(f"Debug - Task type: {type(task)}")
                                print(f"Debug - Task attributes: {dir(task)}")
                                if hasattr(task, '__dict__'):
                                    print(f"Debug - Task dict: {task.__dict__}")
                                return f"Received response but couldn't parse content. Task: {str(task)}"
                        else:
                            # Handle direct response objects
                            response_text = str(response_item)
                            if response_text and response_text != "None":
                                return response_text
                    
                    return f"Received {len(responses)} responses but couldn't extract text content"
                    
                except Exception as e:
                    return f"Error parsing response: {str(e)}. Raw responses: {str(responses)}"
            
            return 'No response received'

# Create client instance
a2a_client = A2AClient()
print('✅ A2A client updated with enhanced response parsing')

✅ A2A client updated with enhanced response parsing


## 测试单个代理

让我们分别测试每个代理，看看它们的工作方式。


In [20]:
# Test Currency Exchange Agent
# 测试货币代理
async def test_currency_agent():
    """Test the currency exchange agent.
    测试货币兑换代理"""
    print("\n🔍 Testing Currency Exchange Agent")
    print("="*50)
    
    response = await a2a_client.send_message(
        'http://localhost:10020',
        'What is the exchange rate from USD to EUR and JPY?' # USD到EUR和JPY的汇率是多少？
    )
    
    print("User: What is the exchange rate from USD to EUR and JPY?")
    print("\nCurrency Agent Response:")
    print(response)

await test_currency_agent()


🔍 Testing Currency Exchange Agent
INFO:     127.0.0.1:52268 - "POST / HTTP/1.1" 200 OK


2026-05-03 18:48:22,538 - INFO - httpx - HTTP Request: POST http://localhost:10020 "HTTP/1.1 200 OK"
2026-05-03 18:48:22,539 - INFO - a2a.client.client_task_manager - New task created with id: caebfc4f-ec13-41fe-92aa-b5ccf9219575


2026-05-03 18:48:24,782 - INFO - httpx - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-03 18:48:24,786 - INFO - semantic_kernel.connectors.ai.open_ai.services.open_ai_handler - OpenAI usage: CompletionUsage(completion_tokens=69, prompt_tokens=362, total_tokens=431, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=256))
2026-05-03 18:48:24,787 - INFO - semantic_kernel.connectors.ai.chat_completion_client_base - processing 2 tool calls in parallel.
2026-05-03 18:48:24,788 - INFO - semantic_kernel.kernel - Calling CurrencyPlugin-get_exchange_rate function with args: {"currency_from": "USD", "currency_to": "EUR", "date": "latest"}
2026-05-03 18:48:24,790 - INFO - semantic_kernel.functions.kernel_function - Function CurrencyPlugin-get_exchange_rate invoking.
2026-05-03 18:48:25,662 - INFO - httpx - HTTP Request: GET https://api.frankfurter.app/latest?from=USD&to=EUR "H

User: What is the exchange rate from USD to EUR and JPY?

Currency Agent Response:
It seems there's an issue with the API we're using to fetch the exchange rates, as it's returning a redirect response which is not expected. The service might have changed its URL structure. 

To get the most accurate and up-to-date exchange rates from USD to EUR and JPY, I recommend checking a reliable financial news site, a banking app, or a dedicated currency converter website. If you need, I can try again later, or you can provide me with another preferred source for the exchange rate information.


In [21]:
# Test Activity Planner Agent
# 测试活动代理
async def test_activity_agent():
    """Test the activity planner agent."""
    print("\n🔍 Testing Activity Planner Agent")
    print("="*50)
    
    response = await a2a_client.send_message(
        'http://localhost:10021',
        'Plan a one-day itinerary for Paris including must-see attractions'  # 为巴黎规划一日游，包括必看景点
    )
    
    print("User: Plan a one-day itinerary for Paris including must-see attractions")
    print("\nActivity Agent Response:")
    print(response)

await test_activity_agent()


🔍 Testing Activity Planner Agent
INFO:     127.0.0.1:52307 - "POST / HTTP/1.1" 200 OK


2026-05-03 18:48:53,513 - INFO - httpx - HTTP Request: POST http://localhost:10021 "HTTP/1.1 200 OK"
2026-05-03 18:48:53,514 - INFO - a2a.client.client_task_manager - New task created with id: c8344896-9867-4ba7-b541-d2af2a27e83a
2026-05-03 18:49:47,304 - INFO - httpx - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-03 18:49:47,307 - INFO - semantic_kernel.connectors.ai.open_ai.services.open_ai_handler - OpenAI usage: CompletionUsage(completion_tokens=1066, prompt_tokens=62, total_tokens=1128, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=0))


User: Plan a one-day itinerary for Paris including must-see attractions

Activity Agent Response:
Certainly! Here’s a detailed, one-day itinerary for Paris that includes some of the city's must-see attractions. This plan is designed to maximize your time and provide a balanced mix of iconic sites, cultural experiences, and local flavors.

### Morning: Explore the Heart of Paris
**8:00 AM - Breakfast at Boulangerie Moderne (14 Rue de l'Exposition, 75007 Paris)**
- Start your day with a typical French breakfast. Enjoy a croissant, pain au chocolat, or a fresh baguette with jam and butter, accompanied by a café au lait.
- **Tip:** Arrive early to avoid lines and enjoy a quiet, authentic Parisian morning.

**9:00 AM - Visit the Eiffel Tower (Champ de Mars, 5 Avenue Anatole France, 75007 Paris)**
- Head to the Eiffel Tower, one of the most famous landmarks in the world. Consider purchasing skip-the-line tickets in advance to save time.
- **Tip:** If you prefer a less crowded experience, tak

## 测试旅行管理器（协调器）

现在让我们测试协调其他代理的主要旅行管理器代理。


In [22]:
# Test Travel Manager with comprehensive request

# 测试旅行管理器
async def test_travel_manager():
    """Test the travel manager orchestrating multiple agents."""
    print("\n🔍 Testing Travel Manager Agent (Orchestrator)")
    print("="*50)
    
    response = await a2a_client.send_message(
        'http://localhost:10022',
        'I am planning a trip to Tokyo. I have 1000 USD to exchange. What is the current exchange rate to JPY and what activities do you recommend for a 2-day visit?'  
        # 我正在计划去东京旅行。我有1000美元要兑换。当前兑日元的汇率是多少？你推荐我在东京进行哪些活动，适合两天的行程？
    )
    
    print("User: I am planning a trip to Tokyo. I have 1000 USD to exchange.")
    print("      What is the current exchange rate to JPY and what activities")
    print("      do you recommend for a 2-day visit?")
    print("\nTravel Manager Response:")
    print(response)

await test_travel_manager()


🔍 Testing Travel Manager Agent (Orchestrator)
INFO:     127.0.0.1:52410 - "GET /.well-known/agent-card.json HTTP/1.1" 200 OK


2026-05-03 18:50:04,201 - INFO - httpx - HTTP Request: GET http://localhost:10022/.well-known/agent-card.json "HTTP/1.1 200 OK"


INFO:     127.0.0.1:52410 - "POST / HTTP/1.1" 200 OK


2026-05-03 18:50:04,205 - INFO - httpx - HTTP Request: POST http://localhost:10022 "HTTP/1.1 200 OK"
2026-05-03 18:50:04,206 - INFO - a2a.client.client_task_manager - New task created with id: 612176c6-1459-4a45-b64b-bfa29a5958a6
2026-05-03 18:50:08,375 - INFO - httpx - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-03 18:50:08,377 - INFO - semantic_kernel.connectors.ai.open_ai.services.open_ai_handler - OpenAI usage: CompletionUsage(completion_tokens=85, prompt_tokens=681, total_tokens=766, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=0))
2026-05-03 18:50:08,377 - INFO - semantic_kernel.connectors.ai.chat_completion_client_base - processing 2 tool calls in parallel.
2026-05-03 18:50:08,378 - INFO - semantic_kernel.kernel - Calling CurrencyExchangeAgent-CurrencyExchangeAgent function with args: {"messages": "I have 1000 USD to exchange. What is the current exch

User: I am planning a trip to Tokyo. I have 1000 USD to exchange.
      What is the current exchange rate to JPY and what activities
      do you recommend for a 2-day visit?

Travel Manager Response:
For your 1000 USD, there is currently a technical issue with the currency exchange rate API, and I'm unable to fetch the latest rates from USD to JPY. As a workaround, I recommend checking a reliable financial news site, using a trusted currency converter website, or contacting your bank for the most up-to-date exchange rates. Once you have the current rate, you can calculate the amount in JPY based on 1000 USD.

As for your 2-day visit to Tokyo, here’s a suggested itinerary:

### Day 1: Exploring Modern and Traditional Tokyo
- **Morning**: Start at Asakusa, visit the Senso-ji Temple, and walk through Nakamise Shopping Street.
- **Late Morning**: Head to Ueno, explore Ueno Park, and visit the Tokyo National Museum or the Ueno Zoo.
- **Lunch**: Try a local restaurant in Ueno for some authe

## 互动测试

尝试输入您自己的旅行规划问题吧！


In [23]:
# 交互式测试
async def interactive_test():
    """Interactive testing function."""
    print("\n🎯 Interactive Travel Planning Assistant")
    print("="*50)
    print("Available agents:")
    print("1. Currency Exchange Agent (port 10020) - Currency conversions")
    print("2. Activity Planner Agent (port 10021) - Travel recommendations")
    print("3. Travel Manager Agent (port 10022) - Comprehensive planning")
    print("\nExample queries:")
    print("- 'Convert 500 EUR to GBP'")
    print("- 'Plan a romantic dinner in Rome'")
    print("- 'I need help planning a budget trip to Seoul with 2000 USD'")
    
    # You can modify this query to test different scenarios
    # 我下周要去伦敦，预算1500 EUR。GBP的汇率是多少？我应该参观哪些主要景点？
    user_query = "I'm visiting London next week with a budget of 1500 EUR. What's the exchange rate to GBP and what are the top attractions I should visit?"
    
    print(f"\nYour query: {user_query}")
    print("\nProcessing with Travel Manager...")
    
    response = await a2a_client.send_message(
        'http://localhost:10022',
        user_query
    )
    
    print("\nResponse:")
    print(response)

await interactive_test()


🎯 Interactive Travel Planning Assistant
Available agents:
1. Currency Exchange Agent (port 10020) - Currency conversions
2. Activity Planner Agent (port 10021) - Travel recommendations
3. Travel Manager Agent (port 10022) - Comprehensive planning

Example queries:
- 'Convert 500 EUR to GBP'
- 'Plan a romantic dinner in Rome'
- 'I need help planning a budget trip to Seoul with 2000 USD'

Your query: I'm visiting London next week with a budget of 1500 EUR. What's the exchange rate to GBP and what are the top attractions I should visit?

Processing with Travel Manager...
INFO:     127.0.0.1:52518 - "POST / HTTP/1.1" 200 OK


2026-05-03 18:51:28,789 - INFO - httpx - HTTP Request: POST http://localhost:10022 "HTTP/1.1 200 OK"
2026-05-03 18:51:28,790 - INFO - a2a.client.client_task_manager - New task created with id: e45be567-4263-4acb-81ae-200f41b4e399
2026-05-03 18:51:32,496 - INFO - httpx - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-03 18:51:32,497 - INFO - semantic_kernel.connectors.ai.open_ai.services.open_ai_handler - OpenAI usage: CompletionUsage(completion_tokens=103, prompt_tokens=674, total_tokens=777, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=0))
2026-05-03 18:51:32,497 - INFO - semantic_kernel.connectors.ai.chat_completion_client_base - processing 2 tool calls in parallel.
2026-05-03 18:51:32,498 - INFO - semantic_kernel.kernel - Calling CurrencyExchangeAgent-CurrencyExchangeAgent function with args: {"messages": "I have a budget of 1500 EUR for my trip to London. C


Response:
It seems there's a temporary issue with fetching the current exchange rate from EUR to GBP due to a redirect error. I recommend checking a reliable financial news site or using a trusted online currency converter for the most up-to-date exchange rates.

Regarding the top attractions in London, here are some must-see places you should consider visiting:

1. **The British Museum** - Home to a vast collection of world art and artifacts.
2. **The Tower of London** - A historic castle known for its rich history and the Crown Jewels.
3. **Buckingham Palace** - The official residence of the British monarch; don't miss the Changing of the Guard ceremony.
4. **The London Eye** - A giant Ferris wheel offering panoramic views of the city skyline.
5. **The Tate Modern** - A modern art gallery featuring works from artists like Picasso, Warhol, and Hockney.
6. **The Natural History Museum** - Famous for its exhibition of dinosaur skeletons.
7. **The Victoria and Albert Museum (V&A)** - Th

## 概要

恭喜你！你已经成功构建了一个多代理旅行规划系统，使用了以下技术：

### 使用的技术：
- **Semantic Kernel** - 用于构建智能代理
- **Azure OpenAI** - 提供大语言模型功能
- **A2A Protocol** - 用于标准化代理通信
- **Uvicorn** - 用于运行本地 A2A 服务器

### 你学到了什么：
1. **代理创建**：使用 Semantic Kernel 构建专门的代理
2. **A2A 集成**：将 SK 代理封装为兼容 A2A 协议
3. **代理编排**：使用管理代理协调多个专业代理
4. **实时服务**：集成外部 API（如 Frankfurter 提供的货币汇率）
5. **本地部署**：在单个笔记本中运行多个 A2A 服务器

### 下一步：
- 将代理部署到 Azure 容器实例或 Azure Functions
- 添加更多专门代理（如航班预订、酒店推荐）
- 实现代理记忆功能，以支持上下文感知的对话
- 为生产环境部署添加认证和安全功能
- 为旅行规划系统创建一个网页界面

### 此架构的主要优势：
- **模块化**：每个代理可以独立开发和部署
- **可扩展性**：代理可以根据需求进行扩展
- **可重用性**：代理可以在不同应用中重复使用
- **互操作性**：A2A 协议允许与不同框架的代理集成



---

**免责声明**：  
本文档使用AI翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于关键信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
